In [1]:
from pathlib import Path
import pandas as pd

# Display settings (optional)
pd.set_option('display.max_colwidth', 120)

# How many rows to show when displaying DataFrames in the notebook
pd.set_option('display.max_rows', 200)  # increase if you want more
pd.set_option('display.min_rows', 50)

In [2]:
# Choose the input JSON file (JSON Whole Model export)
file_name = "ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json"

# Default: workspace-root/JSON Whole Model/<file_name> (works when notebook is in PyDataTransform/)
input_path = (Path('..') / 'JSON Whole Model' / file_name).resolve()
if not input_path.exists():
    input_path = (Path('JSON Whole Model') / file_name).resolve()

assert input_path.exists(), f"File not found: {input_path}"

# 1) Copy the JSON being read into JSON_Edit
json_edit_dir = (Path('..') / 'JSON_Edit').resolve()
if not json_edit_dir.exists():
    json_edit_dir = (Path('JSON_Edit')).resolve()
json_edit_dir.mkdir(parents=True, exist_ok=True)

copied_path = json_edit_dir / input_path.name
copied_path.write_text(input_path.read_text(encoding='utf-8'), encoding='utf-8')
print(f"Copied source JSON to: {copied_path}")

# Work on the copied JSON in JSON_Edit
df = pd.read_json(copied_path)
working_json_path = copied_path
df.shape

Copied source JSON to: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json


(180, 4)

#### Model element Name counts
This notebook loads a `JSON Whole Model/*.json` export and prints a table with **Name**, **DbId**, **GUID**, and the **total count of that Name** across the model (duplicates included).

In [3]:
def _extract_guid(props):
    # `props` is the element's `Properties` array.
    # We look for the entry with displayName == 'GUID' (case-insensitive).
    if not isinstance(props, list):
        return None
    for item in props:
        if not isinstance(item, dict):
            continue
        display_name = str(item.get('displayName', '')).strip()
        if display_name.lower() == 'guid':
            return item.get('value')
    return None

def _strip_guid_prefix(name):
    if pd.isna(name):
        return name
    text = str(name)
    if '_' not in text:
        return text

    prefix, rest = text.split('_', 1)
    # Remove prefixes like "1JNL9441111_" / "1JNL9442575_" (or similar ID-like prefixes)
    if len(prefix) >= 8 and any(ch.isdigit() for ch in prefix):
        return rest
    return text

# 2 & 3) Clean the Name column by removing GUID-like prefixes
df['Name'] = df['Name'].apply(_strip_guid_prefix)

# Save amended JSON into JSON_Edit (does not change original source file)
df.to_json(working_json_path, orient='records', force_ascii=False, indent=2)
print(f"Updated JSON written to: {working_json_path}")

# Build GUID + NameCount columns
df['GUID'] = df['Properties'].apply(_extract_guid)
name_counts = df['Name'].value_counts(dropna=False)
df['NameCount'] = df['Name'].map(name_counts)

table = (
    df[['Name', 'DbId', 'GUID', 'NameCount']]
    .sort_values(['Name', 'DbId'], kind='stable')
    .reset_index(drop=True)
)

# 4) Print out the updated table
rows_to_show = 200  # set to None to show all rows (can be slow/huge)
table if rows_to_show is None else table.head(rows_to_show)

Updated JSON written to: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json


,Name,DbId,GUID,NameCount
0,A-1JNL9322340 - Pyramid,5,8d3567ff-612b-3878-a628-4b9d838df628,1
1,A-Cable Ladder 90 450,14,c9232a6d-75c5-3739-a3c2-3777ac67e597,3
2,A-Cable Ladder 90 450,15,3d931300-8be8-3a23-a464-9cedb7d98460,3
3,A-Cable Ladder 90 450,19,19f22aef-af24-3a1c-b63f-1703e94e50ba,3
4,A-Cable Ladder MVS,12,167265d4-c3e5-3071-b189-599004f7b9f3,9
5,A-Cable Ladder MVS,13,366c5df3-80dd-34c1-95f6-f69c55125c61,9
6,A-Cable Ladder MVS,16,5b00d1af-e807-3f5e-bc90-b4ab1b39b423,9
7,A-Cable Ladder MVS,17,69eed907-29dc-3444-89dd-6e9fe484ce1b,9
8,A-Cable Ladder MVS,18,97558d3b-c88d-3c4c-b66f-b2109faa27c0,9
9,A-Cable Ladder MVS,20,a378d9c0-1349-32e1-9dad-1b5310fafdc9,9


In [7]:
# Export the table
out_dir = input_path.parent
# csv_path = out_dir / f"{input_path.stem}_name_table.csv"
xlsx_path = out_dir / f"{input_path.stem}_name_table.xlsx"

# table.to_csv(csv_path, index=False, encoding='utf-8-sig')
# print("Wrote CSV:", csv_path)

# Excel export requires openpyxl (recommended)
try:
    import openpyxl  # noqa: F401
    table.to_excel(xlsx_path, index=False)
    print("Wrote Excel:", xlsx_path)
except ImportError:
    print("Excel export skipped: package 'openpyxl' is not installed.")
    print("Run: pip install openpyxl  (or use notebook package install), then re-run this cell.")

Wrote Excel: C:\Git\APS-IFC\JSON Whole Model\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002_name_table.xlsx
